# FinIA-Flex — Paso 3: Construcción del Índice Vectorial (RAG)

**Proyecto:** FinIA-Flex — Copiloto Financiero y de Control de Costos para manufactura
**Contexto académico:** Caso Práctico Unidad 1, materia Generative IA — Maestría en Ciencia
de Datos y Analítica Visual, Instituto Europeo de Posgrado

---

## Objetivo del notebook

Construir el índice vectorial que permite al sistema recuperar información relevante de los
documentos de políticas internas (Retrieval-Augmented Generation, RAG) antes de generar un
reporte ejecutivo de variación presupuestal.

## Contenido

1. Instalación de dependencias.
2. Carga de los documentos fuente (Markdown y PDF).
3. División de los documentos en fragmentos (`chunking`).
4. Generación de embeddings.
5. Creación de la base vectorial (ChromaDB).
6. Prueba de búsqueda semántica.
7. Persistencia de la base vectorial para el siguiente paso del pipeline.

## Nota sobre el formato de los documentos fuente

Los documentos de política se distribuyen en formato **Markdown**, simulando el resultado del
paso de preprocesamiento (conversión PDF/Word → texto estructurado) que en un pipeline de
producción se realizaría con herramientas como `PyPDFLoader` o `Docx2txtLoader`. Este enfoque
permite concentrar el prototipo en las técnicas evaluadas en el caso práctico — RAG,
prompting y fine-tuning — sin dedicar tiempo adicional a la extracción de texto desde
formatos binarios, que es un paso de infraestructura y no de modelado.

Para demostrar que el pipeline sí admite formatos de origen distintos, la Sección 3 de este
notebook incluye la carga de una versión en PDF de uno de los documentos, usando
`PyPDFLoader`.

**Decisión técnica:** se utiliza **ChromaDB** como base vectorial en lugar de FAISS, por su
simplicidad de uso en notebooks (persistencia en disco sin configuración adicional) y porque
el volumen de documentos de este prototipo (5 documentos, ~35 fragmentos) no requiere las
optimizaciones de rendimiento que FAISS ofrece a mayor escala.


## 1. Instalación de dependencias

Si Colab reinicia el entorno de ejecución tras instalar `chromadb`, basta con ejecutar nuevamente las celdas desde este punto.

In [ ]:
!pip install -q langchain langchain-community langchain-chroma langchain-text-splitters chromadb sentence-transformers pypdf tiktoken
print("Instalación completa.")

## 2. Carga de los documentos fuente

Los documentos deben colocarse en una carpeta de Google Drive con la siguiente estructura:

```
FinIA-Flex/
└── rag_docs/
    ├── 01_Politica_Gasto_Aprobaciones.md
    ├── 01_Politica_Gasto_Aprobaciones.pdf   <- versión PDF del mismo documento
    ├── 02_Principios_Costeo_Variaciones.md
    ├── 03_Formato_Reporte_Ejecutivo.md
    └── 04_Buenas_Practicas_Optimizacion_Costos.md
```

Como alternativa, los archivos pueden cargarse directamente en la sesión de Colab (panel
Archivos), sin necesidad de montar Google Drive; en ese caso no persisten al cerrar la sesión.


In [ ]:
# Montaje de Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ruta de la carpeta de documentos. Modificar si la estructura de carpetas es distinta.
RAG_DOCS_PATH = "/content/drive/MyDrive/FinIA-Flex/rag_docs"

# Alternativa sin Google Drive (carga manual en el panel Archivos de Colab):
# RAG_DOCS_PATH = "/content/rag_docs"

import os
archivos = os.listdir(RAG_DOCS_PATH)
print(f"Archivos encontrados en {RAG_DOCS_PATH}:")
for a in archivos:
    print(" -", a)

### 2.1 Carga de documentos Markdown

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader_md = DirectoryLoader(
    RAG_DOCS_PATH,
    glob="*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documentos_md = loader_md.load()
print(f"Documentos Markdown cargados: {len(documentos_md)}")
for d in documentos_md:
    print(" -", d.metadata["source"], f"({len(d.page_content)} caracteres)")

### 2.2 Carga de documento en formato PDF

Esta celda demuestra que el pipeline admite formatos distintos a Markdown, cargando la
versión PDF de la Política de Gasto y Aprobaciones (POL-FIN-001) con `PyPDFLoader`. En un
escenario real, todos los documentos de origen llegarían en este tipo de formato (PDF o
Word) y pasarían por este mismo tipo de loader antes del `chunking`.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

ruta_pdf = os.path.join(RAG_DOCS_PATH, "01_Politica_Gasto_Aprobaciones.pdf")

loader_pdf = PyPDFLoader(ruta_pdf)
documentos_pdf = loader_pdf.load()

print(f"Páginas cargadas desde PDF: {len(documentos_pdf)}")
print("\n--- Extracto de la primera página ---")
print(documentos_pdf[0].page_content[:600])

### 2.3 Consolidación de documentos

Para evitar duplicar el contenido de la Política de Gasto (ya presente en formato Markdown),
en este prototipo se conserva únicamente la versión Markdown en el índice final, y el PDF
cargado en la sección anterior se usa solo como demostración del soporte multi-formato. En un
caso con documentos distintos en cada formato, ambas listas se combinarían directamente.


In [ ]:
# Documentos que se indexarán: los 4 documentos en Markdown.
# (documentos_pdf queda disponible como evidencia de soporte multi-formato, sin duplicar contenido)
documentos = documentos_md
print(f"Total de documentos a indexar: {len(documentos)}")

## 3. División de los documentos en fragmentos (chunking)

Se utiliza `RecursiveCharacterTextSplitter` respetando la estructura de encabezados Markdown
(`##`, `###`), de forma que cada fragmento conserve el contexto completo de una sección — por
ejemplo, que el umbral de $50,000 MXN no quede separado de la categoría "Mantenimiento" a la
que aplica.

**Decisión técnica:** `chunk_size=500` con `chunk_overlap=80`. Se eligió un tamaño de
fragmento relativamente pequeño porque los documentos de política contienen secciones cortas
y densas; un fragmento más grande mezclaría reglas de categorías distintas (por ejemplo,
Materia Prima y Mantenimiento) en un mismo bloque, reduciendo la precisión de la recuperación.

**Nota de compatibilidad:** en versiones recientes de LangChain, los divisores de texto
(`text splitters`) se distribuyen en un paquete independiente, `langchain-text-splitters`,
en lugar de estar incluidos directamente en el paquete `langchain`. Por ello la importación
correcta es `from langchain_text_splitters import RecursiveCharacterTextSplitter`, y no
`from langchain.text_splitter import ...` (ruta usada en versiones anteriores de la
librería, que ya no se resuelve correctamente).


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " "],
)

fragmentos = splitter.split_documents(documentos)
print(f"Total de fragmentos generados: {len(fragmentos)}")
print("\n--- Ejemplo de fragmento ---")
print(fragmentos[0].page_content)
print("\nFuente:", fragmentos[0].metadata["source"])

## 4. Generación de embeddings y creación de la base vectorial

Se utiliza un modelo de embeddings local y gratuito (`all-MiniLM-L6-v2`, de
`sentence-transformers`), lo que evita depender de una API de pago en esta etapa del pipeline.
Este modelo es independiente del LLM que se conectará más adelante para la generación de
texto (Paso 5) — ambos procesos no requieren usar el mismo proveedor.


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

CHROMA_PATH = "/content/finia_flex_chroma_db"

vectorstore = Chroma.from_documents(
    documents=fragmentos,
    embedding=embeddings,
    persist_directory=CHROMA_PATH,
)

print(f"Base vectorial creada con {vectorstore._collection.count()} fragmentos indexados.")

## 5. Prueba de búsqueda semántica

Las siguientes preguntas de prueba están relacionadas directamente con los tres escenarios
del dataset simulado (Paso 1), y permiten verificar que el sistema recupera el fragmento
correcto de política para cada caso, y no texto genérico.


In [ ]:
preguntas_prueba = [
    "¿Qué umbral de gasto en mantenimiento requiere aprobación gerencial?",
    "¿Qué se debe hacer si un centro de costo tiene sobrecosto 3 meses seguidos?",
    "¿Un ahorro grande siempre es una buena noticia?",
    "¿Cómo se debe estructurar un reporte ejecutivo de variación de presupuesto?",
]

for pregunta in preguntas_prueba:
    print("=" * 80)
    print("PREGUNTA:", pregunta)
    resultados = vectorstore.similarity_search(pregunta, k=2)
    for i, r in enumerate(resultados, start=1):
        print(f"\n--- Resultado {i} (fuente: {r.metadata['source'].split('/')[-1]}) ---")
        print(r.page_content[:400])
    print()

## 6. Persistencia de la base vectorial

`persist_directory` guarda automáticamente el índice en disco. Al copiar la carpeta a Google
Drive se conserva entre sesiones de Colab, y queda disponible para el Paso 5, donde este RAG
se conecta con el LLM.


In [ ]:
import shutil

DRIVE_BACKUP_PATH = "/content/drive/MyDrive/FinIA-Flex/finia_flex_chroma_db"
shutil.copytree(CHROMA_PATH, DRIVE_BACKUP_PATH, dirs_exist_ok=True)
print("Base vectorial respaldada en:", DRIVE_BACKUP_PATH)

---
## Resumen técnico (Paso 3)

**Proceso realizado:** construcción de un índice vectorial (RAG) a partir de los documentos
de políticas internas de FlexParts Manufacturing MX, usando LangChain, ChromaDB y embeddings
locales de `sentence-transformers`. Se incluyó además la carga de un documento en formato
PDF mediante `PyPDFLoader`, para validar que el pipeline admite múltiples formatos de origen.

**Decisiones técnicas documentadas:**
- ChromaDB en lugar de FAISS, por simplicidad de uso y bajo volumen de documentos.
- Tamaño de fragmento (`chunk_size=500`, `overlap=80`) ajustado para preservar el contexto de
  cada regla de política sin mezclar categorías distintas.
- Embeddings locales y gratuitos (`all-MiniLM-L6-v2`), sin dependencia de una API de pago en
  esta etapa.

**Evidencia generada:** los resultados de la Sección 5 muestran la recuperación correcta de
los fragmentos de las políticas POL-FIN-001 (umbral de aprobación y regla de variación
sostenida) y POL-FIN-002 (clasificación de ahorro atípico).

**Siguiente paso:** Paso 4 — diseño del prompt maestro que combina los fragmentos recuperados
con los datos del dataset simulado para generar el reporte ejecutivo.
